# 🏗️ POC 1: CI Pipeline — Build & Test a Python ML App

**What you'll learn:**
- Build a simple ML prediction API (FastAPI)
- Write tests with pytest
- Containerize with Docker
- Create a Harness CI Pipeline (YAML)

```
Clone Code → pip install → pytest → docker build → docker push
```

## Step 1: Build the ML Prediction App

We'll create a simple sentiment analysis API using scikit-learn.

In [1]:
# Step 1A: Train a simple ML model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pickle, os

# Training data
texts = [
    "I love this product", "This is amazing", "Great quality",
    "Best purchase ever", "Wonderful experience", "Highly recommend",
    "Terrible product", "Worst experience", "Very disappointing",
    "Waste of money", "Horrible quality", "Never buying again",
]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]  # 1=positive, 0=negative

# Build pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression())
])
model.fit(texts, labels)

# Test predictions
test_texts = ["I love it", "This is terrible"]
predictions = model.predict(test_texts)
for text, pred in zip(test_texts, predictions):
    print(f"  '{text}' → {'😊 Positive' if pred == 1 else '😞 Negative'}")

# Save model
os.makedirs('model_artifacts', exist_ok=True)
with open('model_artifacts/sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("\n✅ Model saved to model_artifacts/sentiment_model.pkl")

  'I love it' → 😊 Positive
  'This is terrible' → 😊 Positive

✅ Model saved to model_artifacts/sentiment_model.pkl


In [2]:
# Step 1B: Create the FastAPI app (this is what we'll deploy)

app_code = '''
"""Sentiment Analysis API — Built for Harness CI/CD POC"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pickle, os, datetime

app = FastAPI(title="ML Sentiment API", version="1.0.0")

# Load model
MODEL_PATH = os.getenv("MODEL_PATH", "model_artifacts/sentiment_model.pkl")
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    text: str
    sentiment: str
    confidence: float

@app.get("/health")
def health():
    return {"status": "healthy", "timestamp": str(datetime.datetime.now())}

@app.get("/health/ready")
def readiness():
    return {"status": "ready", "model_loaded": model is not None}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    if not req.text.strip():
        raise HTTPException(400, "Text cannot be empty")
    pred = model.predict([req.text])[0]
    prob = model.predict_proba([req.text])[0]
    return PredictResponse(
        text=req.text,
        sentiment="positive" if pred == 1 else "negative",
        confidence=round(float(max(prob)), 3)
    )

@app.get("/")
def root():
    return {"message": "ML Sentiment API", "docs": "/docs"}
'''

os.makedirs('app', exist_ok=True)
with open('app/main.py', 'w') as f:
    f.write(app_code)
print("✅ Created app/main.py (FastAPI server)")

✅ Created app/main.py (FastAPI server)


: 

## Step 2: Write Tests (pytest)

**Why tests matter for CI:**
- If tests fail → Pipeline stops → Bad code never ships
- Harness runs `pytest` automatically on every code push

In [3]:
# Step 2: Create test files

test_code = '''
"""Tests for Sentiment API — Run by Harness CI"""
import pickle, pytest

# --- Unit Tests: Model ---
def test_model_loads():
    with open("model_artifacts/sentiment_model.pkl", "rb") as f:
        model = pickle.load(f)
    assert model is not None

def test_positive_prediction():
    with open("model_artifacts/sentiment_model.pkl", "rb") as f:
        model = pickle.load(f)
    assert model.predict(["I love this product"])[0] == 1

def test_negative_prediction():
    with open("model_artifacts/sentiment_model.pkl", "rb") as f:
        model = pickle.load(f)
    assert model.predict(["Terrible experience"])[0] == 0

def test_batch_prediction():
    with open("model_artifacts/sentiment_model.pkl", "rb") as f:
        model = pickle.load(f)
    texts = ["Great!", "Awful!"]
    preds = model.predict(texts)
    assert len(preds) == 2

# --- API Tests (using FastAPI TestClient) ---
from fastapi.testclient import TestClient
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "app"))
from main import app

client = TestClient(app)

def test_health():
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "healthy"

def test_readiness():
    r = client.get("/health/ready")
    assert r.status_code == 200

def test_predict_positive():
    r = client.post("/predict", json={"text": "I love this"})
    assert r.status_code == 200
    assert r.json()["sentiment"] in ["positive", "negative"]

def test_predict_empty_text():
    r = client.post("/predict", json={"text": "  "})
    assert r.status_code == 400
'''

os.makedirs('tests', exist_ok=True)
with open('tests/test_app.py', 'w') as f:
    f.write(test_code)
print("✅ Created tests/test_app.py")

✅ Created tests/test_app.py


: 

In [4]:
# Step 2B: Create requirements.txt

requirements = """fastapi==0.104.1
uvicorn==0.24.0
scikit-learn==1.3.2
pydantic==2.5.0
pytest==7.4.3
httpx==0.25.2
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)
print("✅ Created requirements.txt")

✅ Created requirements.txt


: 

## Step 3: Dockerfile

**Multi-stage Docker build:**
- Stage 1 (builder): Install everything, run tests
- Stage 2 (production): Only what's needed to run (small image)

In [5]:
# Step 3: Create Dockerfile

dockerfile = '''# ---- Stage 1: Builder (run tests here) ----
FROM python:3.10-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
RUN pytest tests/ --verbose

# ---- Stage 2: Production (small image) ----
FROM python:3.10-slim
WORKDIR /app
RUN adduser --disabled-password --gecos '' appuser
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
COPY model_artifacts/ ./model_artifacts/
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)
print("✅ Created Dockerfile (multi-stage build)")

✅ Created Dockerfile (multi-stage build)


: 

## Step 4: Harness CI Pipeline YAML

This is the **heart of Harness engineering** — the pipeline definition.

```
Pipeline: Build ML App
├── Stage 1: Build & Test (CI)
│   ├── pip install
│   ├── pytest
│   └── docker build + push
└── Notifications (Slack on failure)
```

In [6]:
# Step 4: Generate Harness CI pipeline YAML
import yaml

pipeline = {
    "pipeline": {
        "name": "ML App CI Pipeline",
        "identifier": "ml_app_ci_pipeline",
        "projectIdentifier": "ml_platform",
        "orgIdentifier": "default",
        "properties": {
            "ci": {
                "codebase": {
                    "connectorRef": "github_connector",
                    "repoName": "your-ml-repo",
                    "build": "<+input>"
                }
            }
        },
        "stages": [
            {
                "stage": {
                    "name": "Build and Test",
                    "identifier": "build_and_test",
                    "type": "CI",
                    "spec": {
                        "cloneCodebase": True,
                        "platform": {"os": "Linux", "arch": "Amd64"},
                        "runtime": {"type": "Cloud", "spec": {}},
                        "execution": {
                            "steps": [
                                {
                                    "step": {
                                        "type": "Run",
                                        "name": "Install Dependencies",
                                        "identifier": "install_deps",
                                        "spec": {
                                            "shell": "Sh",
                                            "command": "pip install -r requirements.txt"
                                        }
                                    }
                                },
                                {
                                    "step": {
                                        "type": "Run",
                                        "name": "Run Tests",
                                        "identifier": "run_tests",
                                        "spec": {
                                            "shell": "Sh",
                                            "command": "pytest tests/ --verbose --tb=short"
                                        }
                                    }
                                },
                                {
                                    "step": {
                                        "type": "BuildAndPushDockerRegistry",
                                        "name": "Build and Push Image",
                                        "identifier": "build_push",
                                        "spec": {
                                            "connectorRef": "dockerhub_connector",
                                            "repo": "myuser/ml-sentiment-api",
                                            "tags": ["<+pipeline.sequenceId>", "latest"]
                                        }
                                    }
                                }
                            ]
                        }
                    }
                }
            }
        ]
    }
}

# Save YAML
os.makedirs('.harness', exist_ok=True)
with open('.harness/ci-pipeline.yaml', 'w') as f:
    yaml.dump(pipeline, f, default_flow_style=False, sort_keys=False)

print("✅ Generated .harness/ci-pipeline.yaml")
print("\n📋 Pipeline YAML:")
print("=" * 50)
print(yaml.dump(pipeline, default_flow_style=False, sort_keys=False))

✅ Generated .harness/ci-pipeline.yaml

📋 Pipeline YAML:
pipeline:
  name: ML App CI Pipeline
  identifier: ml_app_ci_pipeline
  projectIdentifier: ml_platform
  orgIdentifier: default
  properties:
    ci:
      codebase:
        connectorRef: github_connector
        repoName: your-ml-repo
        build: <+input>
  stages:
  - stage:
      name: Build and Test
      identifier: build_and_test
      type: CI
      spec:
        cloneCodebase: true
        platform:
          os: Linux
          arch: Amd64
        runtime:
          type: Cloud
          spec: {}
        execution:
          steps:
          - step:
              type: Run
              name: Install Dependencies
              identifier: install_deps
              spec:
                shell: Sh
                command: pip install -r requirements.txt
          - step:
              type: Run
              name: Run Tests
              identifier: run_tests
              spec:
                shell: Sh
               

: 

## Step 5: Connector YAML (GitHub + Docker Hub)

**Connectors** are how Harness talks to external services.

In [7]:
# Step 5: Generate connector configs

github_connector = {
    "connector": {
        "name": "GitHub Connector",
        "identifier": "github_connector",
        "type": "Github",
        "spec": {
            "url": "https://github.com/YOUR_USERNAME",  # ← Change!
            "authentication": {
                "type": "Http",
                "spec": {
                    "type": "UsernameToken",
                    "spec": {
                        "username": "YOUR_USERNAME",
                        "tokenRef": "github_pat_secret"  # Harness Secret
                    }
                }
            }
        }
    }
}

docker_connector = {
    "connector": {
        "name": "Docker Hub Connector",
        "identifier": "dockerhub_connector",
        "type": "DockerRegistry",
        "spec": {
            "dockerRegistryUrl": "https://index.docker.io/v2/",
            "auth": {
                "type": "UsernamePassword",
                "spec": {
                    "username": "YOUR_DOCKERHUB_USER",
                    "passwordRef": "dockerhub_token"  # Harness Secret
                }
            }
        }
    }
}

for name, config in [("github", github_connector), ("dockerhub", docker_connector)]:
    with open(f'.harness/{name}_connector.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ Created .harness/github_connector.yaml")
print("✅ Created .harness/dockerhub_connector.yaml")
print("\n💡 Replace YOUR_USERNAME and YOUR_DOCKERHUB_USER with real values!")

✅ Created .harness/github_connector.yaml
✅ Created .harness/dockerhub_connector.yaml

💡 Replace YOUR_USERNAME and YOUR_DOCKERHUB_USER with real values!


: 

## 🎯 POC 1 Summary

**Files created:**
| File | Purpose |
|------|---------|
| `app/main.py` | FastAPI ML prediction API |
| `tests/test_app.py` | pytest tests (run in CI) |
| `requirements.txt` | Python dependencies |
| `Dockerfile` | Multi-stage container build |
| `.harness/ci-pipeline.yaml` | Harness CI pipeline |
| `.harness/github_connector.yaml` | GitHub connector |
| `.harness/dockerhub_connector.yaml` | Docker Hub connector |

**Key Learnings:**
1. Pipeline = Stages = Steps (hierarchy)
2. Connectors bridge Harness to GitHub/Docker Hub
3. Tests run automatically — if they fail, pipeline stops
4. Multi-stage Docker = fast, small, secure images

➡️ **Next: POC 2 — Deploy this app to Kubernetes!**